In [ ]:
print("hello world")


hello world


 Zadanie 10 – Custom Funkcja Agregująca
Napisz własną funkcję, która oblicza zakres międzykwartylowy (IQR = Q3 - Q1) i zastosuj ją
do grupy.
Dataset: California Housing
Wymagania:
Oczekiwany rezultat:
(średnie)
Napisz funkcję iqr_func(x) obliczającą IQR
Zastosuj ją do MedHouseVal według regionów
Porównaj IQR między regionami
Tabela z IQR dla każdego regionu
Stwórz regiony na podstawie Latitude/Longitude (podziel na siatkę)


In [1]:
import pandas as pd
from sklearn.datasets import fetch_california_housing

# dane
housing = fetch_california_housing(as_frame=True)
df = housing.frame
df_copy = housing.frame.copy()

# podział na 5 regionów wg Latitude
df_copy['Region'] = pd.cut(df_copy['Latitude'], bins=5, labels=['R1', 'R2', 'R3', 'R4', 'R5'])

# funkcja IQR
def iqr_q3q1(x):
    return x.quantile(0.75) - x.quantile(0.25)
# obliczenie iqr do regionu 
df_copy['Region_IQR'] = df_copy.groupby('Region')['MedHouseVal'].transform(iqr_q3q1)

print(df_copy[['Region', 'MedHouseVal', 'Region_IQR']])


      Region  MedHouseVal  Region_IQR
0         R3        4.526     1.79725
1         R3        3.585     1.79725
2         R3        3.521     1.79725
3         R3        3.413     1.79725
4         R3        3.422     1.79725
...      ...          ...         ...
20635     R4        0.781     0.77750
20636     R4        0.771     0.77750
20637     R4        0.923     0.77750
20638     R4        0.847     0.77750
20639     R4        0.894     0.77750

[20640 rows x 3 columns]


11. ✏ Zadanie 11 – Normalizacja Grupowa
Dla datasetu California Housing znormalizuj MedInc (dochód) w każdym regionie do
zakresu [0, 1].
Dataset: California Housing
Wymagania:
Oczekiwany rezultat:
(średnie)
Stwórz regiony (np. na podstawie Latitude > median)
Napisz funkcję normalizacji Min-Max
Użyj .groupby().transform() lub .apply()
Zweryfikuj, że min=0 i max=1 w każdym regionie
DataFrame z kolumną znormalizowaną grupowo

In [51]:
# podział na 2 regiony wg Latitude - mediana powyżej i rowna, poniżej
mediana_latitude = df_copy['Latitude'].median()

df_copy['Region'] = 'Poludnie'
df_copy.loc[df_copy['Latitude'] > mediana_latitude, 'Region'] = 'Polnoc'
df_copy['Region'] = pd.cut(df_copy['Latitude'], bins=2, labels=['R1', 'R2'])
podzial=df_copy['Region'].value_counts() #sprawdzenie podziału
print(podzial)

# zmiana podziału na pół rekordów - Latitude
df_copy = df_copy.sort_values(by='Latitude').reset_index(drop=True)
polowa = len(df_copy) // 2
df_copy['Region'] = 'Poludnie'
df_copy.iloc[polowa:, df_copy.columns.get_loc('Region')] = 'Polnoc'
podzial=df_copy['Region'].value_counts() #sprawdzenie podziału
print(podzial)

# funkcja normalizacji
def normalizuj(x):
    return (x - x.min()) / (x.max() - x.min())

# normalizacja w grupach
df_copy['MedInc_normalized'] = df_copy.groupby('Region')['MedInc'].transform(normalizuj)
print(df_copy)

# weryfikacja - czy w kazdym regionie jest0 i1
verification = df_copy.groupby('Region')['MedInc_normalized'].agg(['min', 'max'])
print('\nWeryfikacja:')
print(verification)


Region
R1    13296
R2     7344
Name: count, dtype: int64
Region
Poludnie    10320
Polnoc      10320
Name: count, dtype: int64
       MedInc  HouseAge  AveRooms  AveBedrms  Population  AveOccup  Latitude  \
0      1.2667       7.0  3.326241   1.053191      1187.0  4.209220     32.54   
1      1.4384       5.0  3.774005   1.100703      3284.0  3.845433     32.55   
2      3.9583       8.0  5.550552   1.033985      4797.0  4.075616     32.55   
3      1.6348      15.0  3.404321   1.000000      2511.0  3.875000     32.55   
4      1.9875      18.0  3.481375   0.916905      1195.0  3.424069     32.56   
...       ...       ...       ...        ...         ...       ...       ...   
20635  2.2340      19.0  5.988353   1.156406      1572.0  2.615641     41.86   
20636  3.1300      18.0  5.829457   1.383721       660.0  2.558140     41.88   
20637  2.1336      19.0  5.893993   1.144876       841.0  2.971731     41.92   
20638  1.8242      18.0  5.945860   1.350318       802.0  2.554140     41.